<a href="https://colab.research.google.com/github/ananya-ranjith/BIS-LAB/blob/main/Cuckoo_Search_and_grey_wolf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import random


NUM_PM = 5
PM_CPU = 100
PM_RAM = 100


NUM_VM = 10

VM_CPU = np.array([20, 30, 15, 40, 25, 10, 35, 20, 15, 30])
VM_RAM = np.array([25, 20, 30, 35, 15, 10, 25, 20, 15, 30])


NESTS = 20
MAX_ITER = 100
PA = 0.25
STEP_SIZE = 0.1



def fitness(solution):

    pm_cpu = np.zeros(NUM_PM)
    pm_ram = np.zeros(NUM_PM)

    for vm in range(NUM_VM):
        pm = int(solution[vm])

        if pm < 0 or pm >= NUM_PM:
            return 1e10

        pm_cpu[pm] += VM_CPU[vm]
        pm_ram[pm] += VM_RAM[vm]


    penalty = 0

    for pm in range(NUM_PM):
        if pm_cpu[pm] > PM_CPU:
            penalty += (pm_cpu[pm] - PM_CPU) ** 2

        if pm_ram[pm] > PM_RAM:
            penalty += (pm_ram[pm] - PM_RAM) ** 2

    if penalty > 0:
        return 1e6 + penalty


    utilization = []

    for pm in range(NUM_PM):
        cpu_util = pm_cpu[pm] / PM_CPU
        ram_util = pm_ram[pm] / PM_RAM

        if pm_cpu[pm] > 0 or pm_ram[pm] > 0:
            utilization.append((cpu_util + ram_util) / 2)


    if len(utilization) == 0:
        return 1e10

    avg_util = np.mean(utilization)
    imbalance = np.mean((np.array(utilization) - avg_util) ** 2)


    active_pms = len(utilization)

    return active_pms + imbalance




def initialize_nests():

    nests = []

    for i in range(NESTS):
        solution = np.random.randint(
            0, NUM_PM, NUM_VM
        )
        nests.append(solution)

    return np.array(nests)




def levy_flight(solution):

    new_solution = solution.copy()

    for i in range(NUM_VM):

        step = np.random.standard_cauchy() * STEP_SIZE

        new_solution[i] = int(
            np.round(solution[i] + step)
        )

        new_solution[i] = np.clip(
            new_solution[i], 0, NUM_PM - 1
        )

    return new_solution




def cuckoo_search():

    nests = initialize_nests()

    fitness_values = np.array([
        fitness(nest) for nest in nests
    ])

    best_index = np.argmin(fitness_values)

    best_solution = nests[best_index].copy()
    best_fitness = fitness_values[best_index]

    for iteration in range(MAX_ITER):

        for i in range(NESTS):


            new_solution = levy_flight(nests[i])

            new_fitness = fitness(new_solution)


            j = random.randint(0, NESTS - 1)


            if new_fitness < fitness_values[j]:

                nests[j] = new_solution
                fitness_values[j] = new_fitness


            if new_fitness < best_fitness:

                best_solution = new_solution.copy()
                best_fitness = new_fitness


        abandon_count = int(PA * NESTS)

        worst_indices = np.argsort(
            fitness_values
        )[-abandon_count:]

        for index in worst_indices:

            nests[index] = np.random.randint(
                0, NUM_PM, NUM_VM
            )

            fitness_values[index] = fitness(
                nests[index]
            )


        best_index = np.argmin(fitness_values)

        if fitness_values[best_index] < best_fitness:

            best_solution = nests[best_index].copy()
            best_fitness = fitness_values[best_index]

        print(
            f"Iteration {iteration + 1}: "
            f"Best Fitness = {best_fitness:.4f}"
        )

    return best_solution, best_fitness




def display_results(solution):

    pm_cpu = np.zeros(NUM_PM)
    pm_ram = np.zeros(NUM_PM)

    print("\nVM ALLOCATION RESULTS")

    for vm in range(NUM_VM):

        pm = int(solution[vm])

        pm_cpu[pm] += VM_CPU[vm]
        pm_ram[pm] += VM_RAM[vm]

        print(
            f"VM {vm + 1} --> PM {pm + 1}"
        )

    print("\nPM RESOURCE UTILIZATION")

    for pm in range(NUM_PM):

        cpu_util = (pm_cpu[pm] / PM_CPU) * 100
        ram_util = (pm_ram[pm] / PM_RAM) * 100

        print(f"\nPhysical Machine {pm + 1}")
        print(f"CPU Usage: {pm_cpu[pm]} / {PM_CPU}")
        print(f"RAM Usage: {pm_ram[pm]} / {PM_RAM}")
        print(f"CPU Utilization: {cpu_util:.2f}%")
        print(f"RAM Utilization: {ram_util:.2f}%")

    print("\n")




if __name__ == "__main__":

    best_solution, best_fitness = cuckoo_search()

    print("\nBest Fitness:", best_fitness)

    display_results(best_solution)


Iteration 1: Best Fitness = 3.0079
Iteration 2: Best Fitness = 3.0079
Iteration 3: Best Fitness = 3.0079
Iteration 4: Best Fitness = 3.0079
Iteration 5: Best Fitness = 3.0013
Iteration 6: Best Fitness = 3.0013
Iteration 7: Best Fitness = 3.0013
Iteration 8: Best Fitness = 3.0013
Iteration 9: Best Fitness = 3.0013
Iteration 10: Best Fitness = 3.0013
Iteration 11: Best Fitness = 3.0013
Iteration 12: Best Fitness = 3.0013
Iteration 13: Best Fitness = 3.0013
Iteration 14: Best Fitness = 3.0013
Iteration 15: Best Fitness = 3.0013
Iteration 16: Best Fitness = 3.0013
Iteration 17: Best Fitness = 3.0013
Iteration 18: Best Fitness = 3.0013
Iteration 19: Best Fitness = 3.0013
Iteration 20: Best Fitness = 3.0013
Iteration 21: Best Fitness = 3.0013
Iteration 22: Best Fitness = 3.0013
Iteration 23: Best Fitness = 3.0013
Iteration 24: Best Fitness = 3.0013
Iteration 25: Best Fitness = 3.0013
Iteration 26: Best Fitness = 3.0013
Iteration 27: Best Fitness = 3.0013
Iteration 28: Best Fitness = 3.0013
I

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# 1. Generate dummy network traffic dataset (replace with NSL-KDD / CICIDS)
X, y = make_classification(
    n_samples=500, n_features=20, n_informative=10, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


# 2. Fitness Function (evaluates selected features)
def fitness(position):
  # Convert continuous wolf position to binary feature mask (0 or 1)
  selected = position > 0.5
  if np.sum(selected) == 0:  # Prevent empty feature set
    return 1.0

  clf = RandomForestClassifier(n_estimators=20, random_state=42)
  clf.fit(X_train[:, selected], y_train)
  acc = accuracy_score(y_test, clf.predict(X_test[:, selected]))

  # Minimize error rate + slight penalty for using too many features
  return 0.99 * (1 - acc) + 0.01 * (np.sum(selected) / len(position))


# 3. Grey Wolf Optimization (GWO)
def gwo_feature_selection(num_wolves=8, max_iter=15):
  dim = X.shape[1]
  wolves = np.random.rand(num_wolves, dim)

  # Initialize Alpha, Beta, Delta wolves
  alpha_pos, beta_pos, delta_pos = np.zeros(dim), np.zeros(dim), np.zeros(dim)
  alpha_score, beta_score, delta_score = float("inf"), float("inf"), float("inf")

  for t in range(max_iter):
    a = 2 - t * (2 / max_iter)  # Linearly decreases from 2 to 0

    for i in range(num_wolves):
      wolves[i] = np.clip(wolves[i], 0, 1)
      score = fitness(wolves[i])

      # Update Alpha, Beta, Delta
      if score < alpha_score:
        delta_score, delta_pos = beta_score, beta_pos.copy()
        beta_score, beta_pos = alpha_score, alpha_pos.copy()
        alpha_score, alpha_pos = score, wolves[i].copy()
      elif score < beta_score:
        delta_score, delta_pos = beta_score, beta_pos.copy()
        beta_score, beta_pos = score, wolves[i].copy()
      elif score < delta_score:
        delta_score, delta_pos = score, wolves[i].copy()

    # Update wolf positions
    for i in range(num_wolves):
      for j in range(dim):
        r1, r2 = np.random.rand(), np.random.rand()
        A1, C1 = 2 * a * r1 - a, 2 * r2
        D_alpha = abs(C1 * alpha_pos[j] - wolves[i, j])
        X1 = alpha_pos[j] - A1 * D_alpha

        r1, r2 = np.random.rand(), np.random.rand()
        A2, C2 = 2 * a * r1 - a, 2 * r2
        D_beta = abs(C2 * beta_pos[j] - wolves[i, j])
        X2 = beta_pos[j] - A2 * D_beta

        r1, r2 = np.random.rand(), np.random.rand()
        A3, C3 = 2 * a * r1 - a, 2 * r2
        D_delta = abs(C3 * delta_pos[j] - wolves[i, j])
        X3 = delta_pos[j] - A3 * D_delta

        wolves[i, j] = (X1 + X2 + X3) / 3.0

  return alpha_pos > 0.5


# 4. Run GWO and Evaluate
best_features = gwo_feature_selection()
print(
    f"Selected {np.sum(best_features)}/{X.shape[1]} features:"
    f" {np.where(best_features)[0]}"
)

final_clf = RandomForestClassifier(n_estimators=50, random_state=42)
final_clf.fit(X_train[:, best_features], y_train)
final_acc = accuracy_score(y_test, final_clf.predict(X_test[:, best_features]))
print(f"IDS Detection Accuracy: {final_acc * 100:.2f}%")


Selected 7/20 features: [ 1  2  5 11 12 13 19]
IDS Detection Accuracy: 90.67%
